# Evaluate CRAFT + TrOCR on IAM Handwriting Dataset
Benchmarks CER and WER on [Teklia/IAM-line](https://huggingface.co/datasets/Teklia/IAM-line) (all splits) using the traditional OCR pipeline: **CRAFT detection → TrOCR recognition**.

> Runtime → Change runtime type → **T4 GPU** (recommended)

In [1]:
# Install dependencies (Colab)
!pip install -q torch torchvision jiwer datasets transformers accelerate opencv-python-headless scikit-image pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.3 MB/s eta 0:00:00


In [3]:
# (Optional) Mount Drive and cache Hugging Face weights so they don't re-download each session
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
print('HF cache:', os.environ['HF_HOME'])

Mounted at /content/drive
HF cache: /content/drive/MyDrive/hf_cache


In [4]:
# (Optional) If you have a Git URL for this repo, you can clone it into /content.
# Leave REPO_GIT_URL empty to skip.
REPO_GIT_URL = ''
REPO_DIR = '/content/Image-of-handwriting-to-docx'

import os
from pathlib import Path

if REPO_GIT_URL:
    if not Path(REPO_DIR).exists():
        !git clone {REPO_GIT_URL} {REPO_DIR}
    else:
        print('Repo already exists at:', REPO_DIR)
else:
    print('Skipping clone (REPO_GIT_URL is empty).')

Skipping clone (REPO_GIT_URL is empty).


In [5]:
# Point this to the repo folder location in your Colab runtime.
# Options:
# 1) If you copied the repo into Drive: set REPO_DIR to that path.
# 2) If you uploaded a zip and extracted it: set REPO_DIR to the extracted folder.
#
# Example (Drive):
REPO_DIR = '/content/drive/MyDrive/cogs181'


import os
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    raise FileNotFoundError(f'Repo folder not found: {repo_path}. Update REPO_DIR above.')

%cd {repo_path}
print('Working directory:', os.getcwd())

/content/drive/MyDrive/cogs181
Working directory: /content/drive/MyDrive/cogs181


In [ ]:
# Run evaluation twice (0 = all samples).
#
# Mode A: CRAFT detection + crop + TrOCR
!python evaluate_ocr.py --use_craft True  --cuda True --trocr_device cuda --num_samples 200 --output_dir Evaluation_CRAFT_TrOCR
#
# Mode B: TrOCR only on full image (no cropping) — recommended for IAM line images
!python evaluate_ocr.py --use_craft False --trocr_device cuda            --num_samples 200 --output_dir Evaluation_TrOCR_full

python3: can't open file '/content/drive/MyDrive/cogs181/evaluate_ocr.py': [Errno 2] No such file or directory


In [ ]:
# Preview per-sample results (both modes)
import pandas as pd

pd.set_option('display.max_colwidth', 80)

for out_dir in ["Evaluation_CRAFT_TrOCR", "Evaluation_TrOCR_full"]:
    print("=" * 60)
    print(out_dir)
    df = pd.read_csv(f"{out_dir}/eval_results.csv")
    print(f"Mean WER: {df['sample_wer'].astype(float).mean():.4f}")
    print(f"Mean CER: {df['sample_cer'].astype(float).mean():.4f}")
    display(df.head(20))

In [ ]:
# Print summaries (both modes)
for out_dir in ["Evaluation_CRAFT_TrOCR", "Evaluation_TrOCR_full"]:
    print("=" * 60)
    print(out_dir)
    with open(f"{out_dir}/eval_summary.txt") as f:
        print(f.read())

In [ ]:
# (Optional) Copy results to Google Drive
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

for out_dir in ["Evaluation_CRAFT_TrOCR", "Evaluation_TrOCR_full"]:
    drive_dest = f"/content/drive/MyDrive/{out_dir}"
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(out_dir, drive_dest)
    print(f"Results saved to Google Drive: {drive_dest}")